# CMME history — Chad demo

Loads the CMME seasonal forecast history zarr and plots a JAS precipitation
forecast over Chad for a single April initialization.

**April init → JAS**: lead 3 = July, lead 4 = August, lead 5 = September.

In [ ]:
import os

import adlfs
import matplotlib.pyplot as plt
import xarray as xr
import zarr

fs = adlfs.AzureBlobFileSystem(
    account_name="imb0chd0dev",
    sas_token=os.environ["DSCI_AZ_BLOB_DEV_SAS_WRITE"],
)
store = zarr.storage.FsspecStore(
    fs, path="projects/ds-cma-datasharing/processed/CMME_history.zarr"
)
ds = xr.open_zarr(store, consolidated=False)
ds

In [ ]:
# Chad bounding box
LAT_MAX, LAT_MIN = 23.5, 7.5
LON_MIN, LON_MAX = 13.5, 24.0

# April 2010 initialization, JAS (leads 3, 4, 5), mean over leads
jas_mean = (
    ds["PREC"]
    .sel(init_time="2010-04", lead=[3, 4, 5])
    .sel(lat=slice(LAT_MAX, LAT_MIN), lon=slice(LON_MIN, LON_MAX))
    .mean("lead")
    .compute()
)
jas_mean

In [ ]:
fig, ax = plt.subplots(figsize=(5, 6))
jas_mean.plot(
    ax=ax,
    cmap="YlGnBu",
    vmin=0,
    cbar_kwargs={"label": "mm/day"},
)
ax.set_title("CMME — April 2010 init, JAS mean precip\nChad")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()